# QML Lab 1 — Data encoding and quantum kernels
### Qiskit Fall Fest · University of Ottawa

**Question:** can the geometry induced by a quantum circuit help a classical classifier?

This hands-on lab accompanies the first four lessons of IBM's
[Quantum Machine Learning course](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning).
The lecture provides the theory; here you will build and inspect the objects it describes.
The examples and exercises are written for this workshop.

**Prerequisites:** Python/NumPy, qubits and measurement, basic circuit construction.
No previous machine-learning coding is assumed. Notebook 06 continues with trainable circuits.

**Suggested teaching time:** 100–120 minutes, including exercises. For a shorter session,
use §§1, 2, 4, and 5, and demonstrate one encoding from §3. The full shot-based matrix in
§6 and the challenges in §7 are extensions. Timings depend on discussion and hardware.

### By the end, you can

1. Split and preprocess data without leaking test information.
2. Explain and implement basis, amplitude, angle, phase, and dense encodings.
3. Turn circuit overlaps into a kernel matrix and train a support-vector machine (SVM).
4. Compare quantum and classical models on the same held-out examples.
5. Account for circuit counts, shots, data-loading cost, and the limits of the result.

| Course lesson | Hands-on section |
|---|---|
| Introduction to QML | §1: scope, workflow, and evidence |
| Classical ML review | §2: splits, preprocessing, and SVM baselines |
| Data encoding | §3: five encodings; §4: circuit-library maps |
| Quantum kernel methods | §§4–6: fidelity, Gram matrices, SVMs, and sampling |
| Variational classifiers and QNNs | Notebook 06 |

## Setup — run this first

Use **Python 3.12 or newer**. Qiskit **2.5.2** was the latest stable release on
**29 September 2026**; the lab pins that release and the tested companion packages.
These notebooks use V2 primitives and the current circuit-library functions.

- **Colab:** select a Python runtime, change `INSTALL_PACKAGES` to `True`, and run the next cell.
- **Local Jupyter:** select your workshop environment. You can install the same dependencies
  with `python -m pip install -r notebooks/requirements-qml.txt` from the repository root.
- After installing into an already running kernel, **restart the kernel**, set
  `INSTALL_PACKAGES = False`, and run the notebook from the top.

All default cells run locally. There are no accounts, tokens, dataset downloads, or paid jobs
in the default path. Run cells in order. Exercises ask you to change a working baseline, so
**Run All** works before you start editing.

Version sources: [Qiskit on PyPI](https://pypi.org/project/qiskit/),
[Qiskit Machine Learning](https://pypi.org/project/qiskit-machine-learning/),
[Qiskit Aer](https://pypi.org/project/qiskit-aer/).

In [ ]:
INSTALL_PACKAGES = False  # Set True once in a fresh Colab/runtime, then restart the kernel.
if INSTALL_PACKAGES:
    import subprocess
    import sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "qiskit[visualization]==2.5.2", "qiskit-aer==0.17.2",
        "qiskit-machine-learning==0.9.1", "qiskit-ibm-runtime==0.50.0",
        "scipy==1.18.1", "scikit-learn==1.9.1",
        "matplotlib==3.11.2", "pandas==3.0.6",
    ])
    print("Installation complete. Restart the kernel before continuing.")

In [ ]:
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.primitives import StatevectorSampler, StatevectorEstimator

SEED = 27
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (6, 4), "font.size": 11})
display(pd.DataFrame({
    "package": ["qiskit", "qiskit-aer", "qiskit-machine-learning", "scikit-learn"],
    "installed": [version(p) for p in
                  ["qiskit", "qiskit-aer", "qiskit-machine-learning", "scikit-learn"]],
}))
if version("qiskit") != "2.5.2":
    print("This environment differs from the tested Qiskit 2.5.2 environment.")

## 1. What is the learning problem? · 5 min

We study **classical data → quantum circuit → classical prediction**. This is often called
CQ learning: the first letter describes the data, the second the processing. CC is ordinary
classical ML; QC learns from quantum data using classical processing; QQ uses quantum
processing of quantum data. These broad labels hide the cost of acquiring and preparing data.

Our task is **supervised binary classification**: learn labels from examples. Regression
predicts numbers; unsupervised learning searches for structure without labels; reinforcement
learning learns actions from rewards. This lab does not implement every learning paradigm.

Two common routes to QML:

- **Quantum kernel:** keep the circuit fixed, use overlaps as similarities, and train a classical SVM.
- **Variational model:** train circuit parameters using a classical optimizer (Notebook 06).

Our workflow is **map → compile → execute → analyze**. The quantum circuit is one part of
this workflow. A large Hilbert space does not by itself establish an advantage: classical
kernels can also represent very large feature spaces. Any useful claim must include data
preparation, measurements, optimization, and a strong classical baseline. Some apparent
quantum speedups can disappear when a comparable classical access model is allowed
(*dequantization*).

Reading: [course introduction](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning/introduction).

## 2. Build a fair classical experiment · 15 min

We use two noisy interleaving curves, generated locally with `make_moons`.
There are **80 examples**: 40 training, 20 validation, and 20 test. These deliberately small
sets keep the quantum computations accessible; their accuracies will have substantial uncertainty.

- **Training:** fit the scaler and model parameters.
- **Validation:** select model settings.
- **Test:** evaluate the selected model once; do not tune from these results.

For encoding, fit a min–max scaler on **training data only**, mapping its range to $[0,\pi]$.
Validation and test points may fall outside that interval. We preserve them rather than
silently refitting or clipping them. The resulting rotation is still valid.

An SVM finds a separating boundary with a margin. The regularization setting `C` trades off
margin width against errors on training examples. A kernel replaces dot products with a
similarity function, allowing a nonlinear boundary. We compare linear and radial-basis
function (RBF) kernels with the same data and validation procedure.

Reading: [classical ML review](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning/classical-ml-review).

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

X_raw, y = make_moons(n_samples=80, noise=0.18, random_state=SEED)
X_dev, X_test_raw, y_dev, y_test = train_test_split(
    X_raw, y, test_size=0.25, stratify=y, random_state=SEED
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=1 / 3, stratify=y_dev, random_state=SEED
)
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)
assert len(X_train) == 40 and len(X_val) == len(X_test) == 20

fig, ax = plt.subplots()
for points, labels, marker, name in [
    (X_train, y_train, "o", "training"), (X_val, y_val, "^", "validation")
]:
    ax.scatter(points[:, 0], points[:, 1], c=labels, cmap="coolwarm",
               vmin=0, vmax=1, marker=marker, edgecolor="black", label=name)
ax.set(xlabel="Scaled feature 0 (radians)", ylabel="Scaled feature 1 (radians)",
       title="Use training and validation data to develop the model")
ax.legend()
plt.show()

In [ ]:
classical_candidates = []
for kernel in ["linear", "rbf"]:
    for C in [0.1, 1.0, 10.0]:
        gammas = ["scale"] if kernel == "linear" else [0.1, 1.0, "scale"]
        for gamma in gammas:
            model = SVC(kernel=kernel, C=C, gamma=gamma).fit(X_train, y_train)
            classical_candidates.append({
                "kernel": kernel, "C": C, "gamma": gamma,
                "validation_accuracy": model.score(X_val, y_val), "model": model,
            })
# Ties keep the first setting; do not use test accuracy to resolve them.
best_classical = {
    kernel: max((r for r in classical_candidates if r["kernel"] == kernel),
                key=lambda r: r["validation_accuracy"])
    for kernel in ["linear", "rbf"]
}
display(pd.DataFrame([{k: v for k, v in row.items() if k != "model"}
                      for row in classical_candidates]))

### Exercise 1 — Spot the leakage

A teammate scales the complete dataset and then splits it. What information has crossed the
train/test boundary? Also explain why selecting `C` from test accuracy is a problem.

<details><summary>Check your reasoning</summary>

The scaler has learned feature extrema from unseen examples. Model selection from test
accuracy also uses test labels to make development decisions. Fit preprocessing only on
training data and choose settings using validation data. With few samples, repeated splits
or nested cross-validation would give a more reliable comparison; reserve that for a later study.

</details>

## 3. Five ways to encode data · 25 min

A feature map $U(x)$ prepares $|\phi(x)\rangle=U(x)|0\cdots0\rangle$.
The representation determines which differences the learner can see.

| Encoding | Our example | Main tradeoff |
|---|---|---|
| Basis | Bit string `101` → $ |101\rangle $ | Quantization and qubit count grow with precision |
| Amplitude | Four real values → two qubits | Unit normalization removes magnitude; generic loading is costly |
| Angle | One value per qubit using $R_y$ | Simple gates; periodic representation |
| Phase | Relative phase using $H$ then $P(x)$ | Phase needs an interference measurement |
| Dense | Two values per qubit using $R_y$ then $R_z$ | Fewer qubits; coordinate singularities and periodicity |

Qiskit displays bit strings as **$q_{n-1}\cdots q_0$**. The rightmost character is qubit 0.
Amplitude encoding stores a normalized vector, not an arbitrary classical array at zero cost.
It cannot distinguish nonzero vectors that differ only by overall scale or global phase.

Reading: [data encoding](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning/data-encoding).

In [ ]:
from qiskit.circuit.library import StatePreparation

# A. Basis encoding: display order q2 q1 q0.
bit_string = "101"
basis = QuantumCircuit(len(bit_string))
for qubit, bit in enumerate(reversed(bit_string)):
    if bit == "1":
        basis.x(qubit)
assert np.isclose(Statevector.from_instruction(basis).probabilities_dict()[bit_string], 1)
display(basis.draw("mpl"))

# B. Amplitude encoding: validate and normalize explicitly.
def amplitude_encoder(values):
    values = np.asarray(values, dtype=complex)
    if values.ndim != 1 or len(values) < 2 or len(values) & (len(values) - 1):
        raise ValueError("Use a one-dimensional vector whose length is a power of two.")
    norm = np.linalg.norm(values)
    if not np.isfinite(norm) or norm == 0:
        raise ValueError("A finite, nonzero vector is required.")
    state = values / norm
    circuit = QuantumCircuit(int(np.log2(len(values))))
    circuit.append(StatePreparation(state), circuit.qubits)
    return circuit, state

amplitude, normalized = amplitude_encoder([1, 2, 3, 4])
np.testing.assert_allclose(Statevector.from_instruction(amplitude).data, normalized, atol=1e-12)
print("Amplitudes:", normalized)
print("Measurement probabilities:", np.abs(normalized) ** 2)
print("Same encoding after multiplying all features by 10:",
      Statevector.from_instruction(amplitude).equiv(
          Statevector.from_instruction(amplitude_encoder([10, 20, 30, 40])[0])))

In [ ]:
# C. Angle encoding: P(1) = sin(theta/2)^2.
angles = np.array([0.4, 1.1])
angle = QuantumCircuit(2)
for q, theta in enumerate(angles):
    angle.ry(theta, q)
print("Angle-encoded probabilities:", Statevector.from_instruction(angle).probabilities_dict())

# D. Phase encoding: computational-basis measurement cannot see this relative phase.
phi = np.pi / 3
phase = QuantumCircuit(1)
phase.h(0)
phase.p(phi, 0)
phase_x_readout = phase.copy()
phase_x_readout.h(0)  # Rotate X measurement into a computational-basis measurement.
print("Phase, Z-basis probabilities:", Statevector.from_instruction(phase).probabilities())
print("Phase, X-basis probabilities:", Statevector.from_instruction(phase_x_readout).probabilities())
np.testing.assert_allclose(Statevector.from_instruction(phase_x_readout).probabilities()[0],
                           (1 + np.cos(phi)) / 2)

# E. Dense encoding: theta and phi set polar and azimuthal angles on a Bloch sphere.
dense = QuantumCircuit(1)
dense.ry(0.9, 0)
dense.rz(1.2, 0)
sv = Statevector.from_instruction(dense)
print("Dense encoding: <X>, <Y>, <Z> =",
      [float(np.real(sv.expectation_value(SparsePauliOp(p)))) for p in ["X", "Y", "Z"]])
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for circuit, ax, title in zip([angle, phase_x_readout, dense], axes,
                             ["Angle", "Phase with X readout", "Dense"]):
    circuit.draw("mpl", ax=ax)
    ax.set_title(title)
plt.show()

### Exercise 2 — What information survives?

1. Set `phi` to 0 and then $\pi$. Which measurement changes?
2. Set the dense encoder's first angle to 0. Can changing its second angle encode information?
3. Try the zero vector in `amplitude_encoder`. Why is it rejected?
4. Compare an angle and that angle plus $2\pi$. Can a state-based learner distinguish them?

<details><summary>Expected observations</summary>

The phase example remains 50/50 in the Z basis; the X-basis distribution changes.
At the north pole, $R_z$ changes only global phase, so the second dense feature is invisible.
The zero vector cannot be normalized into a quantum state. A single rotation shifted by
$2\pi$ prepares the same physical state up to global phase.

</details>

**Resource check:** amplitude encoding needs only $\log_2 d$ qubits for $d$ amplitudes,
but generic state preparation still needs work scaling with $d$. Include normalization,
loading, and repeated preparation in a runtime comparison; do not assume free quantum RAM.

## 4. From feature-map circuits to fidelity · 20 min

Qiskit provides function constructors for parameterized maps. We use the functions
`z_feature_map`, `zz_feature_map`, `pauli_feature_map`, and `efficient_su2`.
`efficient_su2` is usually a trainable ansatz, but its parameters can also hold data.
A parameter becomes a **feature** or a **weight** because of how we use it.

The maps below do not all accept the same number of values. Compare their parameter counts
as well as compiled gates; this is a resource demonstration, not an accuracy ranking.
The two-qubit examples have all-to-all connectivity. A real device may require routing.

In [ ]:
from qiskit.circuit.library import z_feature_map, zz_feature_map, pauli_feature_map, efficient_su2

maps = {
    "Z": z_feature_map(2, reps=1),
    "ZZ": zz_feature_map(2, reps=1, entanglement="linear"),
    "Pauli Z/XX": pauli_feature_map(2, reps=1, paulis=["Z", "XX"]),
    "SU2 as an encoder": efficient_su2(2, su2_gates=["ry", "rz"], reps=1),
}
resource_rows = []
for name, circuit in maps.items():
    # Explicit parameter-to-value mapping avoids assuming a gate traversal order.
    values = np.linspace(0.2, 1.2, circuit.num_parameters)
    bound = circuit.assign_parameters(dict(zip(circuit.parameters, values)))
    compiled = transpile(bound, basis_gates=["rz", "sx", "x", "cx"],
                         optimization_level=1, seed_transpiler=SEED)
    resource_rows.append({
        "map": name, "qubits": circuit.num_qubits, "input_values": circuit.num_parameters,
        "compiled_depth": compiled.depth(), "CX_count": compiled.count_ops().get("cx", 0),
    })
display(pd.DataFrame(resource_rows))
display(maps["ZZ"].draw("mpl"))

### A circuit defines a similarity

For this lab the quantum kernel is the **squared fidelity**

$$K(x,z)=|\langle\phi(z)|\phi(x)\rangle|^2.$$

Prepare $U(x)|0\rangle$, apply $U(z)^\dagger$, then measure. The all-zero probability
is $K(x,z)$. Circuit order matters: the inverse is applied **after** the first encoding.
The raw complex inner product is not the probability.

With $S$ shots, the estimate is $n_0/S$. Its approximate standard error is
$\sqrt{\hat p(1-\hat p)/S}$. Near probabilities 0 or 1, this plug-in estimate can misleadingly
vanish; use a binomial interval when reporting uncertainty.

Reading: [quantum kernel methods](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning/quantum-kernel-methods).

In [ ]:
feature_map = zz_feature_map(2, reps=1, entanglement="linear")
feature_params = list(feature_map.parameters)

def bind_features(point, scale=1.0):
    point = np.asarray(point)
    if point.shape != (len(feature_params),):
        raise ValueError("Feature dimension does not match the circuit.")
    return feature_map.assign_parameters(dict(zip(feature_params, scale * point)))

def overlap_circuit(left, right, scale=1.0):
    circuit = bind_features(left, scale)
    circuit.compose(bind_features(right, scale).inverse(), inplace=True)
    circuit.measure_all()
    return circuit

left, right = X_train[0], X_train[1]
pair = overlap_circuit(left, right)
SHOTS = 1024
sampler = StatevectorSampler(default_shots=SHOTS, seed=SEED)
counts = sampler.run([pair]).result()[0].data.meas.get_counts()
sampled_fidelity = counts.get("00", 0) / SHOTS
sv_left = Statevector.from_instruction(bind_features(left))
sv_right = Statevector.from_instruction(bind_features(right))
exact_fidelity = abs(np.vdot(sv_right.data, sv_left.data)) ** 2
print("Counts:", counts)
print(f"Exact K(x,z)={exact_fidelity:.4f}; sampled estimate={sampled_fidelity:.4f}")
print("Estimated standard error:", np.sqrt(sampled_fidelity * (1 - sampled_fidelity) / SHOTS))
display(pair.draw("mpl", fold=70))

### Exercise 3 — Check the overlap circuit

Replace `right` with `left`, run the overlap cell, and explain the result. Then restore the
original pair and increase `SHOTS`. Does a larger shot count change the exact fidelity?

<details><summary>Check</summary>

For identical points the ideal inverse cancels the encoding, so all counts are zero bit
strings. Increasing shots reduces typical sampling error; it does not change the ideal
probability. A single estimate need not move closer to the exact value every time.

</details>

## 5. Train an SVM with a quantum kernel · 25 min

For $n$ training examples, the **Gram matrix** has shape $(n,n)$.
For $m$ new examples the cross-kernel has shape $(m,n)$, and its **columns must use the
original training order**. A test-to-test matrix cannot be passed to this trained SVM.

We first use exact statevectors to keep model selection fast and make numerical checks
possible. This is a small-system reference implementation: statevectors require memory
exponential in the number of qubits. §6 replaces the selected kernel with shot estimates.

The ideal fidelity kernel is symmetric, has diagonal entries one, and is positive
semidefinite (PSD). It is a valid kernel because it is the Hilbert–Schmidt inner product
between density matrices: $K(x,z)=\mathrm{Tr}(\rho_x\rho_z)$.

We choose the input scale and SVM `C` on validation data. Classical and quantum searches use
the same split, but have different candidate families and costs; record those differences.

In [ ]:
def feature_states(points, scale):
    return np.array([Statevector.from_instruction(bind_features(row, scale)).data
                     for row in points])

def exact_kernel(left_states, right_states):
    # Rows correspond to left inputs; columns to right inputs.
    return np.abs(left_states.conj() @ right_states.T) ** 2

quantum_candidates = []
for scale in [0.25, 0.5, 1.0]:
    train_states = feature_states(X_train, scale)
    val_states = feature_states(X_val, scale)
    K_train = exact_kernel(train_states, train_states)
    K_val = exact_kernel(val_states, train_states)
    for C in [0.1, 1.0, 10.0]:
        model = SVC(kernel="precomputed", C=C).fit(K_train, y_train)
        quantum_candidates.append({
            "scale": scale, "C": C,
            "validation_accuracy": model.score(K_val, y_val), "model": model,
        })
best_quantum = max(quantum_candidates, key=lambda r: r["validation_accuracy"])
selected_scale = best_quantum["scale"]
train_states = feature_states(X_train, selected_scale)
K_train = exact_kernel(train_states, train_states)
K_test = exact_kernel(feature_states(X_test, selected_scale), train_states)

np.testing.assert_allclose(K_train, K_train.T, atol=1e-12)
np.testing.assert_allclose(np.diag(K_train), 1, atol=1e-12)
assert np.linalg.eigvalsh(K_train).min() > -1e-10
assert K_test.shape == (len(X_test), len(X_train))
display(pd.DataFrame([{k: v for k, v in row.items() if k != "model"}
                      for row in quantum_candidates]))
fig, ax = plt.subplots()
plot = ax.imshow(K_train, vmin=0, vmax=1, cmap="viridis")
ax.set(title=f"Training Gram matrix; selected scale={selected_scale}",
       xlabel="Training example", ylabel="Training example")
fig.colorbar(plot, ax=ax, label="Fidelity")
plt.show()

In [ ]:
# Final evaluation: settings are now frozen.
results = []
for name, row in best_classical.items():
    results.append({
        "model": f"Classical {name} SVM", "validation_accuracy": row["validation_accuracy"],
        "test_accuracy": row["model"].score(X_test, y_test),
        "support_vectors": int(row["model"].n_support_.sum()),
    })
quantum_predictions = best_quantum["model"].predict(K_test)
results.append({
    "model": "Exact quantum-kernel SVM", "validation_accuracy": best_quantum["validation_accuracy"],
    "test_accuracy": accuracy_score(y_test, quantum_predictions),
    "support_vectors": int(best_quantum["model"].n_support_.sum()),
})
display(pd.DataFrame(results))
ConfusionMatrixDisplay.from_predictions(y_test, quantum_predictions, display_labels=["class 0", "class 1"])
plt.title("Held-out predictions: quantum-kernel SVM")
plt.show()
print(f"One test error changes accuracy by {100 / len(y_test):.1f} percentage points.")

### Exercise 4 — Interpret the comparison

- Which model has the highest validation accuracy? Does its test score tell the same story?
- What do support vectors represent? Inspect `best_quantum["model"].support_`.
- Propose an experiment that changes **one** of: noise level, feature map, input scale, or sample count.
  Use a new held-out test set for the next study if you have already inspected this one.

<details><summary>Instructor checkpoint</summary>

Small differences on 20 test examples are weak evidence. A quantum model need not beat RBF,
and a better score would establish neither runtime advantage nor scaling advantage. A sound
follow-up includes multiple seeds/splits, uncertainty, resource accounting, and comparable tuning.
For a precomputed kernel, `support_` identifies training rows; `support_vectors_` does not
store the original feature coordinates.

</details>

## 6. Extension — Estimate the whole kernel with shots · 15–20 min

The exact calculation hid measurement cost. Now use the same overlap circuits and frozen
settings with a finite shot count. We exploit training symmetry and set the ideal diagonal
to one, so there are $n(n-1)/2+mn$ circuits for training and test.

For this lab that is **1,580 circuits** and **808,960 shots** at 512 shots/circuit.
This is local ideal sampling, not a noisy-device prediction. On a device, the diagonal may
also need measurement, and calibration, validation, retries, and queue time add costs.

Independent estimates can make a Gram matrix indefinite. We will inspect its eigenvalues
and construct explicit features from its positive eigenspace. For
$K=U\Lambda U^T$, keep positive $\Lambda_+$ and set
$F_{train}=U_+\sqrt{\Lambda_+}$ and
$F_{new}=K_{new,train}U_+ /\sqrt{\Lambda_+}$.
A linear SVM then uses consistent train/new features. This is one practical regularization
choice, not a recovery of the unknown exact kernel. Very small eigenvalues can amplify noise.

Set `RUN_SHOT_KERNEL = True` to run. Leave it false for a short workshop.

In [ ]:
RUN_SHOT_KERNEL = False
KERNEL_SHOTS = 512

def sampled_kernel_matrices(train, new, scale, shots=512, batch_size=100):
    n, m = len(train), len(new)
    train_kernel = np.eye(n)
    new_kernel = np.zeros((m, n))
    # Build and execute in batches rather than holding every circuit in memory.
    pairs = [("train", i, j) for i in range(n) for j in range(i + 1, n)]
    pairs += [("new", i, j) for i in range(m) for j in range(n)]
    # A shared generator advances across circuits, avoiding reused random draws.
    primitive = StatevectorSampler(default_shots=shots, seed=np.random.default_rng(SEED))
    for start in range(0, len(pairs), batch_size):
        group = pairs[start:start + batch_size]
        circuits = [overlap_circuit(train[i] if kind == "train" else new[i], train[j], scale)
                    for kind, i, j in group]
        pubs = primitive.run(circuits).result()
        for (kind, i, j), pub in zip(group, pubs):
            p0 = pub.data.meas.get_counts().get("0" * feature_map.num_qubits, 0) / shots
            if kind == "train":
                train_kernel[i, j] = train_kernel[j, i] = p0
            else:
                new_kernel[i, j] = p0
    return train_kernel, new_kernel


def positive_kernel_features(train_kernel, new_kernel, relative_cutoff=0.01):
    eigenvalues, U = np.linalg.eigh((train_kernel + train_kernel.T) / 2)
    keep = eigenvalues > max(1e-10, relative_cutoff * eigenvalues.max())
    if not keep.any():
        raise ValueError("No usable positive kernel eigenvalues.")
    roots = np.sqrt(eigenvalues[keep])
    return U[:, keep] * roots, (new_kernel @ U[:, keep]) / roots, eigenvalues

n, m = len(X_train), len(X_test)
number_of_circuits = n * (n - 1) // 2 + m * n
print(f"Budget: {number_of_circuits:,} circuits; {number_of_circuits * KERNEL_SHOTS:,} shots")
if RUN_SHOT_KERNEL:
    shot_train, shot_test = sampled_kernel_matrices(X_train, X_test, selected_scale, KERNEL_SHOTS)
    F_train, F_test, eigenvalues = positive_kernel_features(shot_train, shot_test)
    shot_model = SVC(kernel="linear", C=best_quantum["C"]).fit(F_train, y_train)
    print("Smallest sampled eigenvalue:", eigenvalues.min())
    print("Retained features:", F_train.shape[1])
    print("Mean absolute kernel error:", np.mean(np.abs(shot_train - K_train)))
    print("Shot-kernel test accuracy:", shot_model.score(F_test, y_test))
    print("This is a fixed-setting sampling comparison, not another model-selection step.")
else:
    print("Extension skipped. Set RUN_SHOT_KERNEL=True to execute locally.")

## 7. Exit ticket and hackathon extension

Explain to your partner:

1. Why is $|\langle\phi(z)|\phi(x)\rangle|^2$ a measurable probability?
2. Why does the new-example kernel need a column for **each training example**?
3. What gets harder as you increase sample count? What gets harder as you increase qubits?
4. Which encoding choices discard information?
5. What evidence is missing from an accuracy-only claim of quantum advantage?

### Optional project: a useful similarity

Choose one dataset and compare two encodings against linear/RBF baselines. Submit:

- A diagram of the encoding and a reason for choosing it.
- A reproducible split, validation choices, and a frozen test report.
- A resource table: input values, qubits, compiled two-qubit gates, circuits, and shots.
- A paragraph explaining failures and what you would test next.

**Scaling discussion:** the course describes specially constructed learning tasks related to
discrete logarithms where quantum methods have theoretical benefits under stated assumptions.
Those results do not transfer automatically to our tiny classical dataset. Read the assumptions
before drawing a connection to a practical advantage.

For all-zero readout, a toy model with independent per-bit error $e$ gives $(1-e)^n$ probability
of reading a true all-zero string correctly. This is not a full noise model: nonzero strings can
also be misread as zero, and real errors can be correlated. Notebook 06 measures a concrete noise model.

### Continue

Open **06_QML_Variational_Classifiers_and_Neural_Networks.ipynb** to learn circuit parameters
instead of keeping the feature map fixed.

### References

- IBM course: [introduction](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning/introduction),
  [classical ML](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning/classical-ml-review),
  [encoding](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning/data-encoding),
  [kernels](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning/quantum-kernel-methods).
- API: [StatevectorSampler](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.StatevectorSampler),
  [circuit library](https://quantum.cloud.ibm.com/docs/en/api/qiskit/circuit_library).

These workshop exercises are independent learning material; they are not IBM's course assessment.